# Training Callbacks in xaytune

xaytune provides a powerful event-driven callback system that lets you hook into different stages of the training process. Callbacks are functions that run automatically when specific events occur during training.

The callback system supports **10 events** that cover the entire training lifecycle:
- Training lifecycle: `train_start`, `train_end`
- Epoch lifecycle: `epoch_start`, `epoch_end`
- Step lifecycle: `step_start`, `step_end`
- Evaluation: `eval_start`, `eval_end`
- Checkpointing: `checkpoint_saved`
- Error handling: `error`

Use callbacks to:
- Track and log metrics
- Implement custom early stopping
- Monitor training performance
- Send notifications
- Save custom artifacts
- Debug training runs

## Event Reference

| Event | When it fires |
|-------|---------------|
| `train_start` | Once, before the first epoch |
| `train_end` | Once, after all epochs complete |
| `epoch_start` | At the beginning of each epoch |
| `epoch_end` | At the end of each epoch |
| `step_start` | Before each training step |
| `step_end` | After each training step (metrics available) |
| `eval_start` | Before evaluation runs |
| `eval_end` | After evaluation (eval metrics available) |
| `checkpoint_saved` | After a checkpoint is written to disk |
| `error` | When an exception occurs during training |

## Basic Setup

Create a `CallbackManager` and register callbacks using the `@cb.on("event")` decorator:

In [ ]:
from xaytune.trainer.callbacks import CallbackManager, TrainState

cb = CallbackManager()

@cb.on("train_start")
def on_train_start(state):
    print(f"Training started! {state.num_epochs} epochs planned.")

@cb.on("epoch_end")
def on_epoch_end(state):
    loss = state.metrics.get("loss", "N/A")
    print(f"Epoch {state.epoch} complete — loss: {loss}")

@cb.on("train_end")
def on_train_end(state):
    print(f"Training finished at step {state.global_step}.")

## Testing Callbacks Without Training

You can test your callbacks without running actual training by manually creating a `TrainState` and firing events:

In [ ]:
# You can test callbacks without running actual training
state = TrainState(num_epochs=3, epoch=0, global_step=0)

cb.fire("train_start", state)

# Simulate epoch 1
state.epoch = 1
state.global_step = 100
state.metrics["loss"] = 2.34
cb.fire("epoch_end", state)

# Simulate epoch 2
state.epoch = 2
state.global_step = 200
state.metrics["loss"] = 1.87
cb.fire("epoch_end", state)

state.global_step = 300
cb.fire("train_end", state)

## Pattern: Loss Tracking

Use the `step_end` event to collect training metrics over time:

In [ ]:
loss_history = []

tracker = CallbackManager()

@tracker.on("step_end")
def track_loss(state):
    if "loss" in state.metrics:
        loss_history.append({
            "step": state.global_step,
            "loss": state.metrics["loss"],
        })

# Simulate some steps
state = TrainState()
for i in range(5):
    state.global_step = i + 1
    state.metrics["loss"] = 3.0 - (i * 0.4)  # simulated decreasing loss
    tracker.fire("step_end", state)

print("Loss history:")
for entry in loss_history:
    print(f"  Step {entry['step']}: loss={entry['loss']:.4f}")

## Pattern: Training Timer

Track how long training and individual epochs take:

In [ ]:
import time

timer = CallbackManager()
_timings = {}

@timer.on("train_start")
def start_timer(state):
    _timings["train_start"] = time.time()
    _timings["epoch_times"] = []

@timer.on("epoch_start")
def start_epoch_timer(state):
    _timings["epoch_start"] = time.time()

@timer.on("epoch_end")
def end_epoch_timer(state):
    elapsed = time.time() - _timings["epoch_start"]
    _timings["epoch_times"].append(elapsed)
    print(f"Epoch {state.epoch} took {elapsed:.2f}s")

@timer.on("train_end")
def end_timer(state):
    total = time.time() - _timings["train_start"]
    print(f"Total training time: {total:.2f}s")

# Simulate
state = TrainState(num_epochs=3)
timer.fire("train_start", state)
for epoch in range(1, 4):
    state.epoch = epoch
    timer.fire("epoch_start", state)
    time.sleep(0.1)  # simulate training work
    timer.fire("epoch_end", state)
timer.fire("train_end", state)

## Pattern: Custom Early Stopping

Stop training early when validation loss stops improving. Here's a manual implementation:

In [ ]:
# Manual early stopping callback
es_cb = CallbackManager()
best_loss = float("inf")
patience_counter = 0
PATIENCE = 3

@es_cb.on("eval_end")
def check_early_stop(state):
    global best_loss, patience_counter
    current_loss = state.metrics.get("eval_loss", float("inf"))
    if current_loss < best_loss - 0.01:
        best_loss = current_loss
        patience_counter = 0
        print(f"  New best eval_loss: {current_loss:.4f}")
    else:
        patience_counter += 1
        print(f"  No improvement ({patience_counter}/{PATIENCE})")
        if patience_counter >= PATIENCE:
            state.stop_training()
            print("  → Early stopping triggered!")

# Simulate eval runs
state = TrainState()
eval_losses = [2.5, 2.3, 2.1, 2.15, 2.18, 2.20]
for i, loss in enumerate(eval_losses):
    state.global_step = (i + 1) * 100
    state.metrics["eval_loss"] = loss
    print(f"Eval at step {state.global_step}, loss={loss}")
    es_cb.fire("eval_end", state)
    if state.should_stop:
        print(f"\nTraining would stop at step {state.global_step}")
        break

## Built-in Early Stopping Helper

xaytune provides a ready-to-use early stopping implementation:

In [ ]:
from xaytune.trainer.early_stopping import register_early_stopping_callbacks

cb2 = CallbackManager()
register_early_stopping_callbacks(
    callback_manager=cb2,
    patience=3,
    metric="eval_loss",
    min_delta=0.01,
)

# Same simulation as above
state = TrainState()
for i, loss in enumerate([2.5, 2.3, 2.1, 2.15, 2.18, 2.20]):
    state.metrics["eval_loss"] = loss
    cb2.fire("eval_end", state)
    if state.should_stop:
        print(f"Early stopping triggered after eval #{i+1}")
        break
else:
    print("No early stopping triggered")

## Pattern: Combining Multiple Callbacks

A single `CallbackManager` can hold many callbacks. This lets you compose different behaviors:

In [ ]:
cb_all = CallbackManager()
training_log = []

@cb_all.on("train_start")
def log_start(state):
    training_log.append(f"Started training for {state.num_epochs} epochs")

@cb_all.on("step_end")
def log_step(state):
    if state.global_step % 50 == 0:  # log every 50 steps
        training_log.append(f"Step {state.global_step}: loss={state.metrics.get('loss', 'N/A')}")

@cb_all.on("epoch_end")
def log_epoch(state):
    training_log.append(f"Epoch {state.epoch} complete")

@cb_all.on("train_end")
def log_end(state):
    training_log.append(f"Training complete at step {state.global_step}")

# Simulate a full training run
state = TrainState(num_epochs=2)
cb_all.fire("train_start", state)
for epoch in range(1, 3):
    state.epoch = epoch
    cb_all.fire("epoch_start", state)
    for step in range(1, 101):
        state.global_step += 1
        state.metrics["loss"] = 3.0 / (state.global_step ** 0.3)
        cb_all.fire("step_end", state)
    cb_all.fire("epoch_end", state)
cb_all.fire("train_end", state)

print("Training log:")
for entry in training_log:
    print(f"  {entry}")

## Using Callbacks with Real Training

To use callbacks in actual training, pass your `CallbackManager` to `setup_training()`:

In [ ]:
# In a real training run, pass your CallbackManager to setup_training:
#
# from xaytune.recipes.base import setup_training
# from xaytune.config.schema import TrainConfig, ModelConfig, DataConfig, TrainerConfig
#
# cb = CallbackManager()
#
# @cb.on("step_end")
# def track(state):
#     print(f"Step {state.global_step}: loss={state.metrics.get('loss', 'N/A'):.4f}")
#
# components = setup_training(
#     TrainConfig(
#         recipe="finetune",
#         method="lora",
#         model=ModelConfig(name="meta-llama/Llama-3.1-8B"),
#         data=DataConfig(path="data/train.jsonl", format="alpaca"),
#         trainer=TrainerConfig(num_epochs=3, batch_size=4),
#     ),
#     callback_manager=cb,  # <-- pass your callbacks here
# )
#
# state = components.trainer.train(
#     model=components.model,
#     train_dataloader=components.train_dataloader,
# )
print("See the code above for how to wire callbacks into real training.")
print("setup_training() accepts callback_manager= parameter.")
print("Your custom callbacks run alongside built-in ones (progress, checkpointing, eval).")

## Next Steps

- See `06_advanced.ipynb` for more advanced training patterns
- See `02_finetuning.ipynb` for complete training examples
- Check the callback source code at `xaytune/trainer/callbacks.py` for implementation details

Common use cases:
- **Logging to external services**: Fire webhooks or write to databases on `step_end`
- **Custom checkpointing**: Save model snapshots on `eval_end` when metrics improve
- **Progress notifications**: Send Slack/email updates on `epoch_end` or `train_end`
- **Dynamic learning rate**: Adjust optimizer settings based on metrics during training
- **Debugging**: Log detailed state on `error` events for post-mortem analysis